# Ground Control: fine-tune Qwen3-1.7B as the robot's planner

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough), then **Runtime → Run all**.
2. When cell 2 asks, upload `gc_finetune_data.zip` (from `GroundControl/tools/finetune/`).
3. At the end, download `gc-planner-Q4_0.gguf` (about 1.1 GB) or copy it to Google Drive.

Keep the tab open while it trains (free Colab disconnects idle tabs). Training takes roughly 1–2 hours on a free T4 (one epoch); much less on an A100/L4. Nothing here needs any account other than the Google account you're signed in with.

In [ ]:
# 1. GPU, libraries, and the GGUF converter. The converter is set up first so any
#    problem with it shows up now, not after an hour of training.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.56" "trl>=0.19" "peft>=0.15" "datasets>=3.0" "accelerate>=1.0" sentencepiece protobuf
# Colab ships torchao 0.10, which new PEFT versions refuse to import alongside; it isn't needed here.
!pip -q uninstall -y torchao
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!python llama.cpp/convert_hf_to_gguf.py --help > /dev/null && echo "converter OK"
!ls llama.cpp/build/bin/llama-quantize && echo "quantizer OK"


In [ ]:
# 2. Upload the training data (gc_finetune_data.zip -> train.jsonl, val.jsonl)
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall("data")
print(os.listdir("data"))

In [ ]:
# 3. Build prompt/completion pairs with Qwen3's own chat template, in non-thinking mode.
#    On the board llama.cpp runs with reasoning off, so every prompt ends with an empty
#    <think></think> block before the answer. Training uses the identical format.
import json, random
from datasets import Dataset
from transformers import AutoTokenizer

BASE = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(BASE)

def to_pc(messages):
    prompt = tok.apply_chat_template(messages[:-1], add_generation_prompt=True, enable_thinking=False, tokenize=False)
    return {"prompt": prompt, "completion": messages[-1]["content"] + "<|im_end|>"}

def load(path):
    return [to_pc(json.loads(l)["messages"]) for l in open(path)]

train, val = load("data/train.jsonl"), load("data/val.jsonl")

# General instruction data (alpaca-cleaned) so the same weights keep writing well:
# the board uses this one model for planning, composing answers and choreography.
from datasets import load_dataset
alp = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=193)
extra = []
for ex in alp:
    if len(ex["output"]) > 900:
        continue
    user = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    extra.append(to_pc([{"role": "system", "content": "You are a helpful assistant."},
                        {"role": "user", "content": user}, {"role": "assistant", "content": ex["output"]}]))
    if len(extra) >= int(0.2 * len(train)):
        break
train += extra
random.Random(193).shuffle(train)
print(len(train), "train /", len(val), "val examples")
print(train[0]["prompt"][-300:], "\n---\n", train[0]["completion"])

In [ ]:
# 4. LoRA fine-tune (loss on the completion only)
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

# A100/L4 (compute 8.0+): bf16. T4 (7.5): fp16 mixed precision, fp32 weights.
# (torch.cuda.is_bf16_supported() says True on a T4 because it counts emulation.)
bf16 = torch.cuda.get_device_capability()[0] >= 8
try:   # TRL's fast fp32-output matmul needs compute 8.0+; use its plain fallback on a T4
    import trl.kernels.chunked_logprob as _clp
    if not bf16:
        _clp._MM_OUT_DTYPE = False
except ImportError:
    pass
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16 if bf16 else torch.float32)
cfg = SFTConfig(
    output_dir="gc-lora", num_train_epochs=1, per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=15, logging_steps=25,
    eval_strategy="steps", eval_steps=200, save_strategy="no", max_length=1400,
    bf16=bf16, fp16=not bf16, gradient_checkpointing=True, completion_only_loss=True, report_to="none")
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
trainer = SFTTrainer(model=model, args=cfg, train_dataset=Dataset.from_list(train),
                     eval_dataset=Dataset.from_list(val), processing_class=tok, peft_config=lora)
trainer.train()
trainer.save_model("gc-lora")          # the adapter alone (~70 MB), in case a later cell fails

In [ ]:
# 5. Quick check: plan a few commands the training data never contained
SYSTEM = json.loads(open("data/val.jsonl").readline())["messages"][0]["content"]
def plan(cmd):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": cmd}]
    p = tok.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False,
                                return_tensors="pt", return_dict=True).to(trainer.model.device)
    out = trainer.model.generate(**p, max_new_tokens=160, do_sample=False)
    return tok.decode(out[0][p["input_ids"].shape[1]:], skip_special_tokens=True)
for c in ["Type Hi on the LED matrix and then send back on MQTT on topic luca message 'I am done'",
          "turn right 45 degrees", "how many e's are in strawberry?", "what's the weather tomorrow?",
          "make a stick figure dance like a ballet dancer at 120 bpm on the led matrix", "who invented the light bulb?"]:
    print(c, "\n  ->", plan(c))

In [ ]:
# 6. Merge the LoRA into the base weights and save
merged = trainer.model.merge_and_unload()
merged = merged.to(torch.float16)
merged.save_pretrained("merged", safe_serialization=True)
tok.save_pretrained("merged")
!du -sh merged

In [ ]:
# 7. Convert to GGUF and quantize to Q4_0 (same format as the board's current model)
!python llama.cpp/convert_hf_to_gguf.py merged --outtype f16 --outfile gc-planner-f16.gguf
!./llama.cpp/build/bin/llama-quantize gc-planner-f16.gguf gc-planner-Q4_0.gguf Q4_0
!ls -lh gc-planner-Q4_0.gguf


In [ ]:
# 8. Download (about 1.1 GB). If the browser download is slow or fails, run the Drive cell below instead.
from google.colab import files
files.download("gc-planner-Q4_0.gguf")

In [ ]:
# 8b. (alternative) copy to your Google Drive, then download it from drive.google.com
from google.colab import drive
drive.mount("/content/drive")
!cp gc-planner-Q4_0.gguf /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/gc-planner-Q4_0.gguf